# Learning curves from a training log

Four runs of the same model wrote their losses to one table, a row per run, epoch and
split, in whatever order the runs finished. They evaluated every second epoch, and one of
them crashed partway through. You turn the table into the usual figure: the mean loss of
the runs at each epoch with a band of one standard deviation, for training and validation,
and a mark at the best epoch.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
rows = []
for run in range(4):
    last = 20 if run == 2 else 30                     # run 2 crashed after epoch 20
    for epoch in range(2, last + 1, 2):
        fall = 2.0 * np.exp(-epoch / 6)
        rows.append((run, epoch, "train", fall + 0.05 + rng.normal(0, 0.02)))
        rows.append((run, epoch, "val", fall + 0.25 + 0.002 * max(epoch - 12, 0) ** 2 + rng.normal(0, 0.03)))
log = pd.DataFrame(rows, columns=["run", "epoch", "split", "loss"])
log = log.iloc[rng.permutation(len(log))].reset_index(drop=True)   # rows in no particular order
print(log.head())
print("rows:", len(log))

## Part 1 (coding task): Mean and spread at each epoch

Write `epoch_stats(log, split)` for `split` either `"train"` or `"val"`. It returns a
DataFrame with one row per epoch, indexed by the epoch in increasing order, and three
columns:

- `"mean"`: the mean loss over the runs at that epoch,
- `"sd"`: the standard deviation over the runs, dividing by the number of runs, as
  `np.std` does,
- `"n"`: the number of runs that reported that epoch.

In [ ]:
def epoch_stats(log, split):
    """Mean, sd (dividing by the number of runs) and run count of one split, by epoch."""
    # TODO
    raise NotImplementedError

In [ ]:
val = epoch_stats(log, "val")
train = epoch_stats(log, "train")
print(val.round(3).head(12))
for name in ["mean", "sd", "n"]:
    assert name in val.columns, f"the result has no column {name!r}"
assert list(val.index) == list(range(2, 31, 2)), f"the index should be the epochs 2, 4, ..., 30 in order, got {list(val.index)}"
assert list(val["n"]) == [4] * 10 + [3] * 5, f"n should be 4 up to epoch 20 and 3 after it, got {list(val['n'])}"
by_hand = {e: np.array([v for r, ep, s, v in rows if ep == e and s == "val"]) for e in range(2, 31, 2)}
assert np.allclose(val["mean"], [by_hand[e].mean() for e in val.index]), "the means are wrong: average the val rows of each epoch over the runs"
assert not np.allclose(val["sd"], [by_hand[e].std(ddof=1) for e in val.index]), "these are sample standard deviations with n - 1 in the denominator, the pandas default: pass ddof=0"
assert np.allclose(val["sd"], [by_hand[e].std() for e in val.index]), "the standard deviations are wrong"
assert not np.allclose(train["mean"], val["mean"]), "the train and val summaries are the same: keep only the rows of the split asked for"
print("Part 1 passed")

## Part 2 (coding task): The best epoch

Write `best_complete_epoch(summary)`. It returns the epoch, as an `int`, with the lowest mean loss,
counting only the epochs that every run reached, that is the rows whose `n` equals the
largest `n` in the summary. After a run stops, the mean is taken over fewer runs, and a
drop in it can come from losing a run with high losses rather than from training.

In [ ]:
def best_complete_epoch(summary):
    """The epoch with the lowest mean, among the epochs every run reached."""
    # TODO
    raise NotImplementedError

In [ ]:
toy = pd.DataFrame({"mean": [0.90, 0.60, 0.55, 0.40], "n": [3, 3, 3, 2]}, index=[5, 10, 15, 20])
got = best_complete_epoch(toy)
assert got != 2, "that is the position of the row: return the epoch from the index"
assert got != 20, "epoch 20 has only 2 of the 3 runs: count only the epochs every run reached"
assert got == 15, f"the best complete epoch of the small table is 15, got {got}"
assert isinstance(got, int), f"return a plain int, got {type(got).__name__}"
v, t = best_complete_epoch(val), best_complete_epoch(train)
full = val[val["n"] == 4]
assert v == int(full["mean"].idxmin()), f"best validation epoch: got {v}"
assert t == 20, f"the training mean is lowest at epoch 28, which only 3 runs reached: among the epochs all 4 reached the best is 20, got {t}"
print("best validation epoch:", v, " mean there:", round(val.loc[v, "mean"], 3))
print("Part 2 passed")

## Part 3 (coding task): The figure

Write `draw(train, val)`, taking the two summaries, that returns a Figure with one Axes
holding:

- for each split, its mean as a line against the epoch, labeled `"training loss"` or
  `"validation loss"`, and a shaded band from one standard deviation below that mean to
  one above, drawn with `fill_between`,
- a dashed vertical line at the best complete validation epoch, drawn with `ax.axvline`.

Label the x axis `"epoch"` and the y axis `"loss"`, and add a legend.

In [ ]:
def draw(train, val):
    """Mean lines with one-sd bands for both splits, and the best validation epoch."""
    # TODO
    raise NotImplementedError

In [ ]:
train, val = epoch_stats(log, "train"), epoch_stats(log, "val")
fig = draw(train, val)
assert len(fig.axes) == 1, f"the figure should have one Axes, it has {len(fig.axes)}"
ax = fig.axes[0]
by_label = {line.get_label(): line for line in ax.lines}
for name in ["training loss", "validation loss"]:
    assert name in by_label, f"no line is labeled {name!r}"

def band_has(band, x, lo, hi):
    corners = np.concatenate([p.vertices for p in band.get_paths()])
    def on_edge(ys):
        return all(np.any(np.isclose(corners[:, 0], a) & np.isclose(corners[:, 1], b)) for a, b in zip(x, ys))
    return on_edge(lo) and on_edge(hi)

for summary, name in ((train, "training"), (val, "validation")):
    line = by_label[f"{name} loss"]
    xs = np.asarray(line.get_xdata(), dtype=float)
    assert len(xs) == len(summary), f"the {name} line has {len(xs)} points, not one per epoch"
    assert not np.array_equal(xs, np.arange(len(summary))), f"the {name} line is drawn at x = 0, 1, 2, ...: pass the epochs as x"
    assert np.allclose(xs, summary.index), f"the {name} line should run over the epochs 2, 4, ..., 30"
    assert np.allclose(line.get_ydata(), summary["mean"]), f"the {name} line should be its mean loss"
    x, m, sd = summary.index.to_numpy(), summary["mean"].to_numpy(), summary["sd"].to_numpy()
    assert any(band_has(c, x, m - sd, m + sd) for c in ax.collections), f"no shaded band runs from one standard deviation below the {name} mean to one above it at every epoch"
others = [line for line in ax.lines if line.get_label() not in ("training loss", "validation loss")]
assert len(others) == 1, f"expected one more line, the vertical marker, besides the two means, got {len(others)}"
marker = np.asarray(others[0].get_xdata(), dtype=float)
best = best_complete_epoch(val)
assert np.allclose(marker, best), f"the vertical line should sit at epoch {best}, not {marker[0]}"
assert others[0].get_linestyle() == "--", "draw the marker as a dashed line"
assert ax.get_legend() is not None, "add a legend"
entries = [item.get_text() for item in ax.get_legend().get_texts()]
assert "training loss" in entries and "validation loss" in entries, "the legend should list the two curves by their labels"
assert ax.get_xlabel() == "epoch" and ax.get_ylabel() == "loss", "label the axes 'epoch' and 'loss'"
print("Part 3 passed")

## Solution

### Part 1: Mean and spread at each epoch

In [ ]:
def epoch_stats(log, split):
    """Mean, sd (dividing by the number of runs) and run count of one split, by epoch."""
    rows = log[log["split"] == split]
    by_epoch = rows.groupby("epoch")["loss"]
    return pd.DataFrame({
        "mean": by_epoch.mean(),
        "sd": by_epoch.std(ddof=0),
        "n": by_epoch.size(),
    }).sort_index()

**Why it works.** Filtering on the split first keeps the training and validation losses
apart, and `groupby("epoch")` then collects the losses of every run that reported each
epoch, wherever its rows sit in the shuffled table. Grouping sorts the epochs, and
`sort_index` makes that explicit. pandas' `.std()` puts $n - 1$ in the denominator by
default, so `ddof=0` is what matches `np.std`. With 4 runs the difference is a factor of $\sqrt{4/3} \approx 1.15$, and over the 3 runs
that remain after the crash it is $\sqrt{3/2} \approx 1.22$. `size()` counts the rows of each group, which
shows the crash: from epoch 22 on, every number in the summary comes from 3 runs.

### Part 2: The best epoch

In [ ]:
def best_complete_epoch(summary):
    """The epoch with the lowest mean, among the epochs every run reached."""
    complete = summary[summary["n"] == summary["n"].max()]
    return int(complete["mean"].idxmin())

**Why it works.** Keeping the rows with the largest `n` drops every epoch after the crash, and
`idxmin` returns the index label of the smallest mean, which is the epoch itself. `argmin`
would return the position of that row, counting from 0, which is a different number because
the epochs start at 2 and go up in steps of 2. On the training loss the rule matters: the
mean is lowest at epoch 28, but from epoch 22 on it is an average over 3 runs, so the best
epoch that every run reached is 20.

### Part 3: The figure

In [ ]:
def draw(train, val):
    """Mean lines with one-sd bands for both splits, and the best validation epoch."""
    fig, ax = plt.subplots(figsize=(6, 3.5))
    for summary, name in ((train, "training loss"), (val, "validation loss")):
        e = summary.index.to_numpy()
        center, spread = summary["mean"].to_numpy(), summary["sd"].to_numpy()
        ax.plot(e, center, label=name)
        ax.fill_between(e, center - spread, center + spread, alpha=0.25)
    ax.axvline(best_complete_epoch(val), linestyle="--", color="gray")
    ax.set(xlabel="epoch", ylabel="loss")
    ax.legend()
    return fig

**Why it works.** The summaries are indexed by epoch, so `summary.index` is the x data, and
the two lines run from epoch 2 to 30 rather than from 0 to 14. Each band is the region
between `center - spread` and `center + spread` of its own split, so its edges pass through
those values at every epoch, which is what the check reads from the shaded polygon's
corners. A band of the standard error, $\sigma / \sqrt{n}$, or of two standard deviations
would draw a different width and claim a different thing, so the caption or legend should
always say which one is shown. A line `axvline(e)` stores its x data as `[e, e]`. On this
log the validation mean is lowest at epoch 18 and rises after it while the training loss
keeps falling, so the dashed line marks where the model starts to overfit.